# 与 SDPA 的对比与改造路径


## 1. 结构差异总表

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">维度</th><th style="text-align:left;">第 10 章 SDPA</th><th style="text-align:left;">Gated-KV SDPA</th></tr>
<tr><td>KV 遍历</td><td>全部块，固定顺序</td><td>选中块，sel_idx 序</td></tr>
<tr><td>计算量</td><td>O(S²) 全量</td><td>O(S × sel_count × block)；固定 ratio 时仍为 O(S²)</td></tr>
<tr><td>新增段</td><td>—</td><td>门控选择 + 索引紧凑化（SIMT）</td></tr>
<tr><td>数据依赖</td><td>QK/online/PV/acc 之间已有状态依赖</td><td>选择段产物（sel_idx/count）是注意力段的输入</td></tr>
<tr><td>确定性</td><td>计算次序固定；不同实现/调度下浮点结果仍可能有差异</td><td>需 top-k 破平规则保证</td></tr>
<tr><td>性能调优点</td><td>切分/缓冲/流水</td><td>同左 + 选择占比、ratio 超参扫描</td></tr>
</table>


## 2. 改造路径（增量三步）

在第 10 章完整实现补齐并通过验收后再开始，**每步可独立验收**：

1. **步骤 1：注入"块跳过"**——K 块循环加一个 标量掩码判断（这是算法描述，不使用 Python 成员测试；Host 传 [num_blocks] 的 UINT8/INT32 0/1 表，通过 pl.getval 读取）。验证：功能对（选中集合正确时结果 = gather 版 golden），再测量跳过是否带来性能收益。此步把"选择"与"注意力"的接口（选中集合）定义清楚；
2. **步骤 2：选择段上 NPU**——0/1 表换成 SIMT 确定性排名产生的索引，先用独立 Kernel 建立阶段边界，再研究同 Kernel 发布（第 9 章模块）。验证：sel_idx 与 Host 稳定排序取前 k 的结果逐位一致；
3. **步骤 3：布尔判断换查表遍历**——循环变量从"全块号+跳过"换成"选中序号+查表"（11.03 的改法），最大化跳过收益。

> 步骤 1 的 if 版本是很好的中间态：正确性易验证、跳过逻辑独立调试——**增量改造、步步验收**，避免一次性重写。


## 3. 性能视角：选择段的成本边界

选择段本身有成本（仲裁 + 紧凑化），潜在收益来自跳过部分注意力计算。按一次乘加计 2 FLOPs，跳过一个块的 QKᵀ+PV 共省约 **4×TQ×block×H FLOPs**。本章排名选择成本为 O(num_blocks²) 比较；比较次数、FLOPs 和耗时不能直接相减。净收益取决于省下的稠密计算时间是否超过选择、索引访存、同步和启动开销；降低 ratio 也会改变算法输出。教学版应在调优时实测"选择段耗时 vs 注意力段耗时"占比，验证这一边界。


### 在源码中定位改造边界

第 10 章默认传入 `[0,1,…,num_blocks-1]`，第 11 章传入严格排名后的原始块号表。Q 任务映射、m/l/acc 初始化、QK/PV 和最后归一化完全复用；变化集中在“遍历多少块”和“第 j 次实际读哪块”。先用 Host 生成索引检验消费者，再用 NPU 输出替换，可独立判断错误来自哪一侧。

正文三步中的 0/1 跳过版仍是有价值的中间练习；完整参考展示最终的索引查表版。对 causal，不能只修改 K/V load 偏移却继续使用 j×128 做 mask，后者必须使用查表得到的原 k0。


## 4. 课后练习

1. （选择）改造三步的正确顺序是：
   A. 查表 → 布尔跳过 → 选择上 NPU<br/>
   B. 布尔跳过 → 选择上 NPU → 查表<br/>
   C. 选择上 NPU → 查表 → 布尔跳过
2. （计算）block=128、H=64、TQ=256：跳过一块省多少 FLOPs（QK^T + PV）？num_blocks=32 时，能否只凭比较次数与 FLOPs 推出不亏的 ratio？还需要哪些测量？

In [ ]:
!cat ./answer/11.04_answer.txt

## 5. 本节小结

- 差异核心：**遍历从"全块固定序"到"选中块查表序"**，其余 flash 机制不变；
- 改造三步（布尔跳过 → 选择上 NPU → 查表），步步可验收；
- 性能边界：节省计算与新增选择/交接成本的平衡——选择段成本要实测占比。